# Análisis Exploratorio: Titanic Dataset## ObjetivoAnalizar el dataset del Titanic para identificar características asociadas con la supervivencia,realizando el necesario preprocesamiento y limpieza de datos.## DatasetArchivo: `data/dataset.csv`- 891 pasajeros del RMS Titanic- 12 variables incluyendo datos demográficos, clase y información de supervivencia- Fuente: Kaggle Titanic Competition## Exploración Inicial

In [1]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as sns# Cargar datasetdf = pd.read_csv('data/dataset.csv')# Mostrar primeras filasdf.head()

In [2]:
print('=== INFORMACIÓN DEL DATASET ===')print('Número de pasajeros: ' + str(len(df)))print('Número de columnas: ' + str(len(df.columns)))print('Variables disponibles:')for col in df.columns:    print('  - ' + col)print('Tipos de datos:')for col, dtype in df.dtypes.items():    print('  ' + col + ': ' + str(dtype))print('Valores faltantes:')missing = df.isnull().sum()for col, count in missing.items():    if count > 0:        print('  ' + col + ': ' + str(count) + ' valores faltantes')

In [3]:
# Tratar valores faltantes en Agedf['Age'].fillna(df['Age'].median(), inplace=True)# Tratar valores faltantes en Embarkeddf['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)# Tratar valores faltantes en Cabin (crear categoría 'Desconocido')df['Cabin'] = df['Cabin'].notna().map({'True': 'Tiene cabineta', 'False': 'Sin cabineta'})# Crear variable FamilySizedf['FamilySize'] = df['SibSp'] + df['Parch'] + 1# Crear categorías de edaddef categorizar_edad(age):    if age < 16:        return 'Niño'    elif age < 30:        return 'Joven'    elif age < 60:        return 'Adulto'    else:        return 'Adulto mayor'df['EdadCategoria'] = df['Age'].apply(categorizar_edad)# Crear variable Titulo a partir del nombredf['Titulo'] = df['Name'].str.extract(' ([A-Za-z]+)\', expand=False)titulos_comunes = ['Mr', 'Mrs', 'Miss', 'Master']df['Titulo'] = df['Titulo'].apply(lambda x: x if x in titulos_comunes else 'Other')print('Preprocesamiento completado')print('Filas después de limpieza: ' + str(len(df)))

In [4]:
# Análisis: Supervivencia por génerosns.countplot(data=df, x='Sex', hue='Survived')plt.title('Supervivencia por Género')plt.savefig('outputs/resultados/supervivencia_genero.png')plt.show()# Análisis: Supervivencia por clasesns.countplot(data=df, x='Pclass', hue='Survived')plt.title('Supervivencia por Clase')plt.savefig('outputs/resultados/supervivencia_clase.png')plt.show()# Análisis: Supervivencia por edadsns.countplot(data=df, x='EdadCategoria', hue='Survived')plt.title('Supervivencia por Categoría de Edad')plt.savefig('outputs/resultados/supervivencia_edad.png')plt.show()

In [5]:
# Generar reporte de conclusionessupervivencia_genero = df.groupby('Sex')['Survived'].mean().reset_index()supervivencia_genero.columns = ['Sexo', 'Tasa_supervivencia']supervivencia_clase = df.groupby('Pclass')['Survived'].mean().reset_index()supervivencia_clase.columns = ['Clase', 'Tasa_supervivencia']print('Tasa de supervivencia general')print('Tasa de supervivencia por género')print(supervivencia_genero.to_string(index=False))print('Tasa de supervivencia por clase')print(supervivencia_clase.to_string(index=False))# Guardar reportereporte = dict(    total_pasajeros=len(df),    tasa_supervivencia_general=float(df['Survived'].mean()),    supervivencia_por_genero=supervivencia_genero.to_dict(),    supervivencia_por_clase=supervivencia_clase.to_dict(),)with open('../preprocesamiento.json', 'w') as f:    json.dump(reporte, f, indent=2, default=str)print('Reporte guardado en preprocesamiento.json')